# 3AM 非官方复现教程

本 Notebook 覆盖环境检查、权重、数据、预处理、训练、评测和 3D 实例分割。

## 1. 环境检查

In [ ]:
import sys, pathlib
ROOT = pathlib.Path('/Users/lanjie/Proj/3dgs/3am')
sys.path.insert(0, str(ROOT / 'src'))
import three_am
print('three_am', three_am.__version__)

## 2. 下载模型权重

SAM2 权重可直接下载；MUSt3R 若上游未提供固定 URL，请设置 `MUST3R_CHECKPOINT_URL`。

In [ ]:
# !python scripts/download_weights.py --config configs/full_reproduction.yaml

## 3. 下载数据集

需要用户提供官方授权下载命令，脚本从环境变量读取，避免泄露 token。

In [ ]:
# !python scripts/download_datasets.py --config configs/full_reproduction.yaml

## 4. 预处理与 Manifest

将各数据集整理为 `frames/masks/depth/poses/intrinsics` 结构后生成统一 manifest。

In [ ]:
# !python scripts/build_manifest.py --dataset scannetpp --root data/processed/scannetpp --split train --output data/processed/scannetpp_manifest.json

## 5. 单场景可视化

读取 manifest 后检查图片、mask、depth、pose 是否对齐。

In [ ]:
from three_am.data.io import read_manifest
# scenes = read_manifest('data/processed/scannetpp_manifest.json')
# print(scenes[0])

## 6. 启动训练

论文全量训练为 1M iterations；先运行 smoke test 验证 FeatureMerger 与优化器。

In [ ]:
!python scripts/train_3am.py --config configs/full_reproduction.yaml --smoke --iterations 10

## 7. 恢复训练

全量训练阶段应在 CUDA 服务器上恢复 `outputs/checkpoints/3am_full.pt`。

In [ ]:
# !python scripts/train_3am.py --config configs/full_reproduction.yaml

## 8. 2D 评测

对 ScanNet++/Replica 预测 mask 与 GT mask 计算 IoU、Tracking Recall、Accuracy。

In [ ]:
# !python scripts/evaluate_2d.py --pred-dir outputs/pred_masks --gt-dir data/processed/replica/example/masks --output outputs/eval/replica_metrics.json

## 9. 3D 实例分割

SAM2 自动生成 keyframe masks，3AM 传播，depth/pose 反投影，再按 3D/2D overlap 合并。

In [ ]:
# !python scripts/run_3d_instance_segmentation.py --manifest data/processed/replica_manifest.json --checkpoint outputs/checkpoints/3am_full.pt --output-dir outputs/instance3d

## 10. 常见错误排查

- SAM2 import 失败：先运行 `bash scripts/install_external.sh`。
- 数据下载跳过：检查对应环境变量是否设置。
- 全量训练 OOM：降低输入分辨率或使用多卡梯度累积。
- 结果不等于论文：官方 3AM 代码未公开，本仓库是 faithful reimplementation。